In [0]:
#dbutils.widgets.text("race_results_max_ingestion","")
race_results_max_ingestion=dbutils.widgets.get("race_results_max_ingestion")
print(type(race_results_max_ingestion))

In [0]:
class Gold_season_summary():
    main_path="/Volumes/formula1_race/default/formula1/"
    gold_path = "formula1_race_project/gold"
    silver_path = "formula1_race_project/silver"

    def __init__(self,table,max_ingestion_date):
         self.table=table
         self.max_ingestion_date=max_ingestion_date  #latest water mark value for race_results table from Gold_race_results class

    def read_input(self):
        from pyspark.sql.functions import max,col,expr,count,to_timestamp,lit,try_to_timestamp
        race_year_list=list()
        #print the latest water mark value
        print(f"max_ingestion_date:{self.max_ingestion_date}")
        print(f"max_ingestion_date:{type(self.max_ingestion_date)}")

        #fetching incremental race_results data from gold table race_results
        if spark.catalog.tableExists("formula1_race.silver.race_results"):
           Incr_race_results_df= (spark.read.table('formula1_race.silver.race_results')
                                  .filter(col('results_ingestion_date')> self.max_ingestion_date))
           
           #fetching distinct race_year from incremental race_results data
           Incr_race_results_df_list = (Incr_race_results_df
                                   .select(col('race_year')).distinct()
                                   .orderBy(col('race_year').asc()).collect()
                                   )
           #fetching incremental race_results data and printing count of records
           print("season_summary:Incr_race_results_df batch count")
           display(Incr_race_results_df.select(count('*')))
           
           #listing of distinct race_years and printing the list
           race_year_list=[r.race_year for r in Incr_race_results_df_list]
           
        print(f"race_year_list:{race_year_list}")
        return race_year_list
    
    def apply_transformations(self,race_year_list):
        from pyspark.sql.functions import round,col,broadcast,dense_rank,avg,expr,sum,count,min,when
        from pyspark.sql.window import Window
        spec_window=Window.partitionBy("race_year").orderBy(col('total_points').desc())
        
        #fetching only required data from race_results table which is required for aggregation and printing the count of records
        Incr_race_results_df= (spark.read.table('formula1_race.silver.race_results')
                               .filter(col('race_year').isin(race_year_list))
                               )
        print("season_summary:Incr_race_results_df original count")
        display(Incr_race_results_df.select(count('*')))
        
        #aggregating data as per the requirements and printing out sample year data for clarification
        season_summary_df= (Incr_race_results_df.groupBy(col("race_year"),col("driver_name"))
                        .agg(sum(col("result_points")).alias("total_points"),
                        count(col("race_name")).alias("grand_prix_races"),
                        expr("count(case when result_position_order= 1 then result_position_order end)").alias("wins"),
                        min(col("result_position_order")).alias('Best_race_results_position'),
                    count(when(col("result_position") != 0,col("result_position"))).alias("No_of_race_finshes"),
                    count(when(col("result_position") == 0,col("result_position"))).alias("race_does_not_finshes"),
                    count(when(col("result_fastest_lap_rank") == 1, col("result_fastest_lap_rank"))).alias("fastest_laps"),
                    round(avg(col("result_position_order")),2).alias("Avg_race_position")          
                    ).withColumn("position",dense_rank().over(spec_window))
                    .select(col("race_year"),col("driver_name"),col("position"),col("total_points"),
                            col("grand_prix_races"),col("wins"),col("Best_race_results_position"),col("No_of_race_finshes"),col("race_does_not_finshes"),col("fastest_laps"),col("Avg_race_position"))
                         )
        display(season_summary_df.filter((col("race_year")==2018)))
        return season_summary_df
    
    def write_output(self,apply_tran_df):
         # writing those data into gold layer table by partitioning according to filter approache using dynamic partitionOverwriteMode as true with overwritte mode
        (apply_tran_df.write
        .option("partitionOverwriteMode", "dynamic")
        .partitionBy("race_year").mode("overwrite").saveAsTable(f"formula1_race.gold.{self.table}"))
        print("final_count_in table")
        display(spark.sql(f'select count(*) from formula1_race.gold.{self.table}'))
        print("Data write into gold season_summary table is Done")
    
    def process(self):
        print("Started gold-ingestion-season_summary in runing....")
        race_year_list=self.read_input() #return list of distinct race_years
        apply_tran_df=self.apply_transformations(race_year_list) #return aggregated data
        self.write_output(apply_tran_df)#write data into gold table                    
        

In [0]:
Gold_season_summary_instance = Gold_season_summary("season_summary",race_results_max_ingestion)
Gold_season_summary_instance .process()
print("Successfully Gold_season_summary is ran")